In [377]:
import os
import glob
import pandas as pd
import warnings

In [379]:
# Path to folder containing .xlsx files
DATA_DIR = "./data/related/outcomes_endpoints"


In [381]:
# Required columns in each Excel file
REQUIRED_COLS = [
    'Test Name',
    'Measure',
    'Outcome Domain',
    'Subdomain'
]

# Pattern to match .xlsx files
file_pattern = os.path.join(DATA_DIR, '*.xlsx')
excel_files = glob.glob(file_pattern)

if not excel_files:
    warnings.warn(f"No Excel files found in {DATA_DIR}")

# List to collect dataframes
dfs = []
processed_count = 0 
for filepath in excel_files:
    if 'Endpoints_' not in filepath or ('~' in filepath):
        continue
    processed_count += 1
    print(filepath)
    # Read the Excel file into a DataFrame
    df = pd.read_excel(filepath)

    filename = os.path.basename(filepath)

    # Check for required columns
    missing = set(REQUIRED_COLS) - set(df.columns)
    if missing:
        warnings.warn(
            f"File '{filename}' is missing columns: {', '.join(missing)}",
            UserWarning
        )
        # Continue processing; missing columns will result in NaNs

    # Normalize Test Name: lowercase and strip whitespace
    if 'Test Name' in df.columns:
            df['Test Name'] = (
                df['Test Name']
                .astype(str)
                .str.strip()
                .str.lower()
                # Normalize dashes to standard hyphen
                .str.replace(r'[‐–—]', '-', regex=True)
                # Remove trailing words 'test', 'tests', 'task', or 'tasks'
                #.str.replace(r"\s+(?:test|tests|task|tasks)$", '', regex=True)
            )


    # Extract article key from filename (e.g. Endpoints_shepherdTranslationalAssaysAssessment2016)
    article_key = os.path.splitext(filename)[0].split('_')[-1]

    # Add a column for the source article
    df['Article'] = article_key

    dfs.append(df)

# Concatenate all into one DataFrame
combined = pd.concat(dfs, ignore_index=True)

print("Processed files: ",processed_count)

# Define aggregation functions:
# - For Measure: join unique values
# - Outcome Domain, Subdomain: join unique values
# - Articles: join unique values
agg_funcs = {
    'Measure': lambda x: '; '.join(sorted(set(x.dropna()))),
    'Outcome Domain': lambda x: '; '.join(sorted(set(x.dropna()))),
    'Subdomain': lambda x: '; '.join(sorted(set(x.dropna()))),
    'Article': lambda x: '; '.join(sorted(set(x.dropna())))
}

# Group by normalized Test Name and aggregate
harmonized_raw = combined.groupby('Test Name', as_index=False).agg(agg_funcs)

# Optional: reorder columns
columns = ['Test Name', 'Measure', 'Outcome Domain', 'Subdomain', 'Article']
harmonized_raw = combined.groupby('Test Name', as_index=False).agg(agg_funcs)

# Save to Excel or CSV
output_path = os.path.join(DATA_DIR, 'harmonized_assays_raw.xlsx')
harmonized_raw.to_excel(output_path, index=False)

print(f"Harmonized DataFrame saved to {output_path}")


./data/related/outcomes_endpoints/Endpoints_osierChronicHistopathologicalBehavioral2015.xlsx
./data/related/outcomes_endpoints/Endpoints_waerzeggersMouseModelsNeurological2010.xlsx
./data/related/outcomes_endpoints/Endpoints_websterUsingMiceModel2014.xlsx
./data/related/outcomes_endpoints/Endpoints_zarrukNeurologicalTestsFunctional2011.xlsx
./data/related/outcomes_endpoints/Endpoints_guevaraSystematicReviewAnimalbased2022.xlsx
./data/related/outcomes_endpoints/Endpoints_javaeedHistologicalStainsPresent2021.xlsx
./data/related/outcomes_endpoints/Endpoints_jonesOutcomeMeasuresRodent2025.xlsx
./data/related/outcomes_endpoints/Endpoints_goldFunctionalAssessmentLongTerm2013.xlsx
./data/related/outcomes_endpoints/Endpoints_alturkistaniHistologicalStainsLiterature2016.xlsx
./data/related/outcomes_endpoints/Endpoints_acikgozOverviewCurrencyUsefulness2022.xlsx
./data/related/outcomes_endpoints/Endpoints_wahlCognitiveBehavioralEvaluation2017.xlsx
./data/related/outcomes_endpoints/Endpoints_tremo

/var/folders/nd/2fzvhsh510gbt9x6z5pdb1gr0000gn/T/ipykernel_48969/966536654.py:32: UserWarning: File 'Endpoints_waerzeggersMouseModelsNeurological2010.xlsx' is missing columns: Subdomain
  warnings.warn(
/var/folders/nd/2fzvhsh510gbt9x6z5pdb1gr0000gn/T/ipykernel_48969/966536654.py:32: UserWarning: File 'Endpoints_meredithBehavioralModelsParkinsons2006.xlsx' is missing columns: Subdomain
  warnings.warn(
/var/folders/nd/2fzvhsh510gbt9x6z5pdb1gr0000gn/T/ipykernel_48969/966536654.py:32: UserWarning: File 'Endpoints_markicevicEmergingImagingMethods2021a.xlsx' is missing columns: Subdomain
  warnings.warn(
/var/folders/nd/2fzvhsh510gbt9x6z5pdb1gr0000gn/T/ipykernel_48969/966536654.py:32: UserWarning: File 'Endpoints_jinNewDevelopmentsWestern2015.xlsx' is missing columns: Measure
  warnings.warn(
/var/folders/nd/2fzvhsh510gbt9x6z5pdb1gr0000gn/T/ipykernel_48969/966536654.py:32: UserWarning: File 'Endpoints_xiongAnimalModelsTraumatic2013.xlsx' is missing columns: Subdomain
  warnings.warn(


Harmonized DataFrame saved to ./data/related/outcomes_endpoints/harmonized_assays_raw.xlsx


In [382]:
harmonized_raw.shape

(494, 5)

In [383]:
harmonized_raw.head()

,Test Name,Measure,Outcome Domain,Subdomain,Article
0,10x genomics chromium,3′-end UMI barcoding in nanoliter droplets ena...,Molecular & Cellular,Single-cell transcriptomics,adilSingleCellTranscriptomicsCurrent2021
1,3-chamber test,Crosses into empty; Crosses into occupied; Lat...,Behavioral,Sociability,harrisonUnifiedBehavioralScoring2020
2,5-choice serial reaction time task (5-csrtt),"Response accuracy, omissions, premature respon...",Behavioral,Attention,shepherdTranslationalAssaysAssessment2016
3,5-choice serial reaction time task (csrtt),"Accuracy, omissions, reaction time, premature ...",Behavioral,Attention & executive function,websterUsingMiceModel2014
4,[^11c]flumazenil pet,GABAA receptor binding potential,Imaging,,waerzeggersMouseModelsNeurological2010


In [384]:
harmonized_raw[harmonized_raw['Outcome Domain']=='Behaviour']

,Test Name,Measure,Outcome Domain,Subdomain,Article


In [385]:
set(harmonized_raw['Outcome Domain'])

{'Behavioral', 'Histology', 'Imaging', 'Molecular & Cellular', 'Physiology'}

## Harmonize ChatGPT help

In [392]:
mapping = {
    # 5-choice serial reaction time task variants
    '5-choice serial reaction time task (5-csrtt)': '5-choice serial reaction time task',
    '5-choice serial reaction time task (csrtt)':  '5-choice serial reaction time task',

    # Adhesive removal
    'adhesive removal':                           'adhesive removal test',
    'bilateral tactile adhesive removal':         'adhesive removal test',

    # Adjusting‐step test variants
    'adjusting steps':                            'adjusting-step test',
    'adjusting-step (stepping) test':             'adjusting-step test',

    # Beam/Balance-beam variants
    'balance beam':                               'beam walking test',
    'beam traversal':                             'beam walking test',
    'beam walking':                               'beam walking test',
    'beam walking test':                          'beam walking test',

    # BOLD fMRI sequence variants
    'bold fmri (gre-epi)':                        'bold fmri',
    'bold fmri (se-epi)':                         'bold fmri',

    # Conditioned place avoidance ≃ aversion
    'conditioned place avoidance (cpa)':          'conditioned place aversion',

    # Cylinder test variants
    'cylinder (forelimb asymmetry) test':         'cylinder test',
    'cylinder spontaneous placement':             'cylinder test',
    'limb use asymmetry (cylinder)':              'cylinder test',

    # Delayed/non-matching to place water maze
    'delay-nonmatching-to-place (dnmtp) water maze':  'delayed matching-to-place water maze',
    'delayed matching-to-place (dmp) water maze':      'delayed matching-to-place water maze',
    'delayed matching/non-matching to position (dmtp/dmts)': 'delayed matching-to-place water maze',

    # Diffusion MRI variants
    'diffusion mri (adc mapping)':                'diffusion mri',
    'diffusion mri (dmri)':                       'diffusion mri',

    # DTI vs DWI
    'diffusion tensor imaging (dti)':             'diffusion tensor imaging',
    'diffusion-weighted imaging (dwi)':           'diffusion tensor imaging',

    # Functional MRI variants
    'functional mri (bold fmri)':                 'functional mri',
    'functional mri (fmri)':                      'functional mri',

    # Foot-fault
    'foot fault':                                 'foot fault test',
    'foot-fault test':                            'foot fault test',

    # Forced swim
    'forced swim task':                           'forced swim test',
    'forced swim test':                           'forced swim test',

    # Fear conditioning
    'fear conditioning':                          'fear conditioning test',
    'fear conditioning (general fc)':             'fear conditioning test',
    'fear conditioning - contextual':             'fear conditioning test',
    'fear conditioning - contextual (fc-c)':      'fear conditioning test',
    'fear conditioning - cued':                   'fear conditioning test',
    'fear conditioning - cued (fc-u)':            'fear conditioning test',
    'fear conditioning test':                     'fear conditioning test',

    # Grid test variants
    'grid task (acute mptp mice)':                'grid test',
    'grid-walk test':                             'grid test',

    # Grip strength
    'grip strength meter test':                   'grip strength test',

    # Grooming
    'grooming':                                   'grooming test',

    # Hargreaves
    'hargreaves (radiant-heat) test':             'hargreaves test',

    # Home-cage monitoring
    'home-cage activity monitoring':              'home cage monitoring',

    # Inclined plane
    'inclined plane':                             'inclined plane test',

    # Intrinsic optical imaging
    'intrinsic optical imaging (ioi)':            'intrinsic optical imaging',

    # Laser speckle contrast imaging
    'laser speckle contrast imaging (lsci)':      'laser speckle contrast imaging',

    # Light–dark box
    'light-dark compartment test':                'light-dark box test',
    'light/dark box':                             'light-dark box test',

    # Limb placement
    'limb-placing function':                      'limb placement test',

    # MRI / MRS
    'magnetic resonance imaging (mri)':           'magnetic resonance imaging',
    'magnetic resonance spectroscopy (mrs)':      'magnetic resonance spectroscopy',
    'magnetic resonance spectroscopy mrs (proton magnetic resonance spectroscopy)': 'magnetic resonance spectroscopy',
    'mr spectroscopy (mrs)':                      'magnetic resonance spectroscopy',

    # Modified Neurological Severity Score
    'modified neurological severity score':               'modified neurological severity score',
    'modified neurological severity score (mnss)':        'modified neurological severity score',
    'modified neurological severity score (mnss) (mouse)': 'modified neurological severity score',
    'modified neurological severity score (mnss) (rat)':   'modified neurological severity score',

    # Morris Water Maze variants
    'morris water maze':                         'morris water maze',
    'morris water maze (mwm)':                   'morris water maze',
    'morris water maze - learning latency':      'morris water maze',
    'morris water maze - probe trial':           'morris water maze',
    'morris water maze - reversal learning':     'morris water maze',
    'morris water maze - swim speed':            'morris water maze',
    'morris water maze - working memory':        'morris water maze',

    # Novel Object Recognition
    'novel object recognition':                  'novel object recognition test',
    'novel object recognition (nor)':            'novel object recognition test',
    'novel object recognition test (nort)':      'novel object recognition test',

    # Open-Field / Open Field
    'open field locomotion':                     'open field test',
    'open field test':                           'open field test',
    'open-field':                                'open field test',
    'open-field activity':                       'open field test',
    'open-field test':                           'open field test',

    # Neurological Severity Score (unmodified)
    'neurological severity score (nss)':         'neurological severity score',
    'neurological scoring':                      'neurological severity score',

    # Passive avoidance
    'passive avoidance':                         'passive avoidance test',

    # Perfusion MRI
    'perfusion mri (cbf-weighted fmri)':         'perfusion mri (cbf-weighted)',
    'perfusion mri (cbv-weighted fmri)':         'perfusion mri (cbv-weighted)',

    # Radial-arm maze variants
    'radial arm maze (ram)':                     'radial arm maze',
    'radial-arm maze':                           'radial arm maze',
    'radial arm water maze (rawm)':              'radial arm water maze',
    'radial-arm water maze (rawm)':              'radial arm water maze',

    # Resting-state fMRI
    'resting-state fmri (bold)':                 'resting-state fmri',

    # Rotarod
    'rotarod':                                   'rotarod test',

    # Single-pellet reaching
    'single pellet reaching':                    'single pellet reaching task',

    # Staircase
    'staircase':                                 'staircase test',
    'staircase test (monkey “hill”/“valley”)':   'staircase test',

    # Social approach/avoidance
    'social approach':                           'social approach test',
    'social avoidance':                          'social avoidance test',

    # T-maze variants
    't-maze/y-maze alternation':                 't-maze',

    # T2-weighted MRI variant
    't2/t2-weighted mri*':                      't2-weighted mri',

    # Three-chamber social approach
    'three-chambered social approach test':      'three-chamber social approach test',

    # What-where-which
    'what-where-which (wwwhich) task':           'what-where-which (wwwhich)',

    # Wheel running
    'wheel running activity':                    'wheel running',

    # Von Frey filament
    'von frey filament test (up/down)':          'von frey filament test',

    # Y-maze alternation
    'y-maze alternation':                        'y-maze',
}


In [393]:
mapping.update({
    k.lower(): v.lower()
    for k, v in {
        # Hematoxylin & Eosin
        'H&E':                                'Hematoxylin & Eosin',
        'Hematoxylin and Eosin':              'Hematoxylin & Eosin',
        'Hematoxylin & Eosin (H&E)':          'Hematoxylin & Eosin',
        # Gram stain
        'Gram Stain':                         'Gram stain',
        "Gram’s Stain":                       'Gram stain',
        # Giemsa stain
        'Giemsa (Romanowsky) Stain':          'Giemsa stain',
        'Giemsa Stain':                       'Giemsa stain',
        # Periodic acid–Schiff
        'Periodic Acid–Schiff (PAS)':         'Periodic acid–Schiff',
        'Periodic acid–Schiff (PAS)':         'Periodic acid–Schiff',
        'Periodic Acid':                      'Periodic acid–Schiff',
        'Schiff Reagent':                     'Periodic acid–Schiff',
        # Silver-based stains
        'Silver Nitrate (Argentaffin) Stain': 'Silver nitrate stain',
        'Silver Nitrate':                     'Silver nitrate stain',
        'Silver Stain (Bielschowsky, etc.)':  'Silver stain',
        # Prussian Blue
        'Prussian Blue Stain':                'Prussian Blue',
        'Prussian Blue':                      'Prussian Blue',
        # Lipid stains
        'Sudan Black':                        'Sudan Black',
        'Oil Red O':                          'Oil Red O',
        # Ammoniacal Carmine
        'Ammoniacal Carmine':                 'Ammoniacal Carmine',
        # Congo Red, Mucicarmine, Sirius Red
        'Congo Red':                          'Congo Red',
        'Mucicarmine':                        'Mucicarmine',
        'Sirius Red':                         'Sirius Red',
        # Nissl / Cresyl Violet
        'Nissl (Cresyl Violet)':              'Nissl stain',
        'Nissl':                              'Nissl stain',
        # Papanicolaou
        'Papanicolaou (Pap)':                 'Papanicolaou stain',
        'Pap':                                'Papanicolaou stain',
        # Acid-fast / fluorescent
        'Ziehl–Neelsen (Acid-Fast) Stain':    'Ziehl–Neelsen stain',
        'Auramine–Rhodamine Fluorescent Stain':'Auramine–Rhodamine stain',
        'Auramine & Rhodamine':               'Auramine–Rhodamine stain',
        # Endospore stains
        'Endospore Stain (Wirtz method)':     'Endospore stain',
        'Endospore Stain – Schaeffer-Fulton Method': 'Endospore stain',
        'Endospore Stain – Dorner Method':    'Endospore stain',
        # Frozen Section
        'Frozen Section H&E':                 'Frozen Section',
        'Frozen Section (Cryostat)':          'Frozen Section',
        # IHC variants
        'Immunoperoxidase IHC':               'Immunohistochemistry (IHC)',
        'Immunofluorescence IHC':             'Immunohistochemistry (IHC)',
        # Tetrachrome
        'Tetrachrome (Alcian Blue + PAS + Hematoxylin + Picric Acid)': 'Tetrachrome',
        # Methylene Blue variants
        'Aniline-dye (Methylene Blue) Stain':  'Methylene Blue',
        'Methylene Blue':                      'Methylene Blue',
        # Western blot variants
        'Western blotting':                    'Western blot',
        'Microfluidic Western blot':           'Western blot',
        'Simple Western (ProteinSimple)':      'Western blot',
        'Multianalyte on-chip Western':        'Western blot',
        'SNAP i.d.':                           'Western blot',
        'iBlot':                               'Western blot',
        # Capillary electrophoresis variants
        'Capillary electrophoresis (CE)':      'Capillary electrophoresis',
        'Capillary zone electrophoresis (CZE)': 'Capillary electrophoresis',
        'Capillary isoelectric focusing (cIEF)': 'Capillary electrophoresis',
        'Capillary gel electrophoresis (CGE)':  'Capillary electrophoresis',
        'Micellar electrokinetic capillary chromatography (MEKC)': 'Capillary electrophoresis',
    }.items()
})

mapping.update({
    k.lower(): v.lower()
    for k, v in {
        # [18F] FDG PET
        '[^18f]fdg pet':                          '[^18f]fdg pet',
        '¹⁸f-fdg pet':                            '[^18f]fdg pet',
        'pet [¹⁸f]-fdg metabolic imaging':        '[^18f]fdg pet',

        # micro-PET
        'micropet':                               'micro-pet',

        # co-registered PET/CT
        'co-registered ct-pet imaging':          'co-registered pet/ct',
        'co-registered pet/ct':                  'co-registered pet/ct',

        # photoacoustic tomography
        'photoacoustic computed tomography':     'photoacoustic tomography',
        'photoacoustic tomography (pat)':        'photoacoustic tomography',

        # magnetic resonance spectroscopy
        'magnetic resonance spectroscopy':       'mr spectroscopy',
        'mr spectroscopy':                       'mr spectroscopy',
        'functional mrs (fmrs)':                 'mr spectroscopy',  # subtype mapping to same base

        # structural / anatomical MRI
        'anatomical ¹h mri':                     'structural mri',
        'structural mri':                        'structural mri',

        # functional MRI (fMRI)
        'functional mri':                        'functional mri',
        'bold fmri':                             'functional mri',
        'task-based fmri':                       'functional mri',
        'resting-state fmri':                    'functional mri',

        # diffusion MRI
        'diffusion mri':                         'diffusion mri',
        'diffusion tensor imaging':              'diffusion mri',

    }.items()
})


In [394]:
# 2) Secondary processing: apply mapping, collect synonyms, and re-group
# Map raw names to canonical
harmonized_raw['Canonical Name'] = harmonized_raw['Test Name'].replace(mapping)
# Keep raw as synonym
harmonized_raw['Synonym'] = harmonized_raw['Test Name']

# Group by Canonical Name
agg2 = {
    'Measure': lambda x: '; '.join(sorted(set(x.dropna()))),
    'Outcome Domain': lambda x: sorted(set(x.dropna()))[0],
    'Subdomain': lambda x: '; '.join(sorted(set(x.dropna()))),
    'Article': lambda x: '; '.join(sorted(set(x.dropna()))),
    'Synonym': lambda x: '; '.join(sorted(set(x.dropna())))
}
final_harmonized = harmonized_raw.groupby('Canonical Name', as_index=False).agg(agg2)

# Reorder columns
cols = ['Canonical Name', 'Synonym', 'Outcome Domain', 'Subdomain', 'Measure', 'Article']
final_harmonized = final_harmonized[cols]

# Save to Excel
output_path = os.path.join(DATA_DIR, 'harmonized_with_synonyms.xlsx')
final_harmonized.to_excel(output_path, index=False)
print(f"Saved final harmonized table with synonyms to {output_path}")

Saved final harmonized table with synonyms to ./data/related/outcomes_endpoints/harmonized_with_synonyms.xlsx


In [396]:
final_harmonized.shape

(407, 6)

In [397]:
final_harmonized.head()

,Canonical Name,Synonym,Outcome Domain,Subdomain,Measure,Article
0,10x genomics chromium,10x genomics chromium,Molecular & Cellular,Single-cell transcriptomics,3′-end UMI barcoding in nanoliter droplets ena...,adilSingleCellTranscriptomicsCurrent2021
1,3-chamber test,3-chamber test,Behavioral,Sociability,Crosses into empty; Crosses into occupied; Lat...,harrisonUnifiedBehavioralScoring2020
2,5-choice serial reaction time task,5-choice serial reaction time task (5-csrtt); ...,Behavioral,Attention; Attention & executive function,"Accuracy, omissions, reaction time, premature ...",shepherdTranslationalAssaysAssessment2016; web...
3,[^11c]flumazenil pet,[^11c]flumazenil pet,Imaging,,GABAA receptor binding potential,waerzeggersMouseModelsNeurological2010
4,[^11c]methionine (met) pet,[^11c]methionine (met) pet,Imaging,,Amino‐acid transport/protein synthesis (tumor ...,waerzeggersMouseModelsNeurological2010


### add GPT synonyms

In [402]:
df_syn_gpt = pd.read_excel(os.path.join(DATA_DIR, 'harmonized_synonyms_GPT.xlsx'))

final_harmonized = final_harmonized.merge(
    df_syn_gpt[['Canonical Name', 'Synonyms_GPT']],
    on='Canonical Name',
    how='left'
)

df_syn_gpt.shape

(390, 2)

In [404]:
final_harmonized[final_harmonized['Synonyms_GPT'].isna()].to_csv("data/related/todo_gpt_synonyms.csv",index=False)

In [406]:
# Merge GPT synonyms into final_harmonized
final_harmonized['Synonym'] = final_harmonized.apply(
    lambda row: '; '.join(
        [row['Synonym']] + ([row['Synonyms_GPT']] if pd.notnull(row['Synonyms_GPT']) else [])
    ), axis=1
)
# Drop the temporary Synonyms_GPT column
final_harmonized = final_harmonized.drop(columns=['Synonyms_GPT'])


In [407]:
new_row = {
    'Canonical Name': 'positron emission tomography',
    'Synonym': 'positron emission tomography; metabolic PET; PET scan; PET imaging; PET image; PET images; PET/CT; PET‐CT; positron emission tomographic scan; positron emission tomographic image; PET tomogram; PET tomograms; PET tomograph; PET tomographs',
    'Outcome Domain': 'Physiology',
    'Subdomain': 'Nuclear Imaging',
    'Measure': 'Standard uptake value (SUV); SUV_max; SUV_mean',
    'Article': 'custom'
}

# Add the row
final_harmonized = pd.concat([final_harmonized, pd.DataFrame([new_row])], ignore_index=True)


In [409]:
new_row = {
    'Canonical Name': 'spectroscopy',
    'Synonym': 'spectral analysis; spectrometric assay; optical spectrometry; spectral measurement; spectral characterization; spectrochemical analysis',
    'Outcome Domain': 'Molecular & Cellular',
    'Subdomain': 'Spectroscopy',
    'Measure': 'Chemical composition analysis via electromagnetic spectra (e.g. NMR-, IR-, Raman- or MS-based)',
    'Article': 'custom'
}

# Add the row
final_harmonized = pd.concat([final_harmonized, pd.DataFrame([new_row])], ignore_index=True)


In [412]:
new_row = {
    'Canonical Name': 'gene expression analysis',
    'Synonym': 'gene expression profiling; transcript quantification; gene expression assay; transcriptional analysis',
    'Outcome Domain': 'Molecular & Cellular',
    'Subdomain': 'Gene expression assay',
    'Measure': 'Fold change; transcript abundance; Ct value; RPKM; TPM; relative expression',
    'Article': 'custom'
}
final_harmonized = pd.concat([final_harmonized, pd.DataFrame([new_row])], ignore_index=True)


In [414]:
final_harmonized

,Canonical Name,Synonym,Outcome Domain,Subdomain,Measure,Article
0,10x genomics chromium,10x genomics chromium,Molecular & Cellular,Single-cell transcriptomics,3′-end UMI barcoding in nanoliter droplets ena...,adilSingleCellTranscriptomicsCurrent2021
1,3-chamber test,3-chamber test; three-chamber social approach ...,Behavioral,Sociability,Crosses into empty; Crosses into occupied; Lat...,harrisonUnifiedBehavioralScoring2020
2,5-choice serial reaction time task,5-choice serial reaction time task (5-csrtt); ...,Behavioral,Attention; Attention & executive function,"Accuracy, omissions, reaction time, premature ...",shepherdTranslationalAssaysAssessment2016; web...
3,[^11c]flumazenil pet,[^11c]flumazenil pet; [^11C]flumazenil PET;11C...,Imaging,,GABAA receptor binding potential,waerzeggersMouseModelsNeurological2010
4,[^11c]methionine (met) pet,[^11c]methionine (met) pet; [^11C]MET PET;C-11...,Imaging,,Amino‐acid transport/protein synthesis (tumor ...,waerzeggersMouseModelsNeurological2010
...,...,...,...,...,...,...
405,¹³c-fmrs,¹³c-fmrs,Imaging,MRI–Spectroscopy,Tracking of ¹³C-labeled substrates to quantify...,justProtonFunctionalMagnetic2021
406,¹⁸f-flt pet,¹⁸f-flt pet,Imaging,PET – Proliferation tracer,Uptake of ¹⁸F-fluorothymidine indicating cellu...,tremoledaImagingTechnologiesBasic2015
407,positron emission tomography,positron emission tomography; metabolic PET; P...,Physiology,Nuclear Imaging,Standard uptake value (SUV); SUV_max; SUV_mean,custom
408,spectroscopy,spectral analysis; spectrometric assay; optica...,Molecular & Cellular,Spectroscopy,Chemical composition analysis via electromagne...,custom


In [417]:
# Save to Excel
output_path = os.path.join(DATA_DIR, 'final_harmonized_with_enriched_synonyms.xlsx')
final_harmonized.to_excel(output_path, index=False)
print(f"Saved enriched harmonized table with synonyms to {output_path}")

Saved enriched harmonized table with synonyms to ./data/related/outcomes_endpoints/final_harmonized_with_enriched_synonyms.xlsx


In [419]:
final_harmonized.to_csv("../08_IE_full_text/data/assay_extraction/assay_final_harmonized_with_enriched_synonyms.csv", index=False)

df_test = pd.read_csv("../08_IE_full_text/data/assay_extraction/assay_final_harmonized_with_enriched_synonyms.csv")

In [421]:
df_test

,Canonical Name,Synonym,Outcome Domain,Subdomain,Measure,Article
0,10x genomics chromium,10x genomics chromium,Molecular & Cellular,Single-cell transcriptomics,3′-end UMI barcoding in nanoliter droplets ena...,adilSingleCellTranscriptomicsCurrent2021
1,3-chamber test,3-chamber test; three-chamber social approach ...,Behavioral,Sociability,Crosses into empty; Crosses into occupied; Lat...,harrisonUnifiedBehavioralScoring2020
2,5-choice serial reaction time task,5-choice serial reaction time task (5-csrtt); ...,Behavioral,Attention; Attention & executive function,"Accuracy, omissions, reaction time, premature ...",shepherdTranslationalAssaysAssessment2016; web...
3,[^11c]flumazenil pet,[^11c]flumazenil pet; [^11C]flumazenil PET;11C...,Imaging,NaN,GABAA receptor binding potential,waerzeggersMouseModelsNeurological2010
4,[^11c]methionine (met) pet,[^11c]methionine (met) pet; [^11C]MET PET;C-11...,Imaging,NaN,Amino‐acid transport/protein synthesis (tumor ...,waerzeggersMouseModelsNeurological2010
...,...,...,...,...,...,...
405,¹³c-fmrs,¹³c-fmrs,Imaging,MRI–Spectroscopy,Tracking of ¹³C-labeled substrates to quantify...,justProtonFunctionalMagnetic2021
406,¹⁸f-flt pet,¹⁸f-flt pet,Imaging,PET – Proliferation tracer,Uptake of ¹⁸F-fluorothymidine indicating cellu...,tremoledaImagingTechnologiesBasic2015
407,positron emission tomography,positron emission tomography; metabolic PET; P...,Physiology,Nuclear Imaging,Standard uptake value (SUV); SUV_max; SUV_mean,custom
408,spectroscopy,spectral analysis; spectrometric assay; optica...,Molecular & Cellular,Spectroscopy,Chemical composition analysis via electromagne...,custom
